#  Teplomer LM92 


[LM92](./doc/lm92.pdf) je 12-bitový teplotný senzor pre teplotný rozsah -25 ... +150°C s rozlíšením 0.0625°C. Teplomer má možnosť nastavenie kritických teplôt a výstup pre vyvolanie prerušenia pri prekročení stanovených teplôt.

##  Pripojenie  

    MICROPY_HW_I2C1_SCL         -> pin_PB8  (STM)
    MICROPY_HW_I2C1_SDA         -> pin_PB9  (STM)
    +5V
    GND

```{figure} ./img/stm32_i2c1.png
:width: 350px
:name: mp_0205f

Pripojenie teplomera LM92
```


##  Knižnica 

```Python
def read_LM92_TR(ic, addr):
    raw = ic.mem_read(2, addr, 0)      # nacitanie 2 byte z registra 0 
    data = (raw[0] << 8) + raw[1]      # konverzia 16 bit format
    td = data >> 3                     # 2'compl b15-b3 temperature
    TEMP = (-(td & 0x1000) | (td & 0xFFF))* 0.0625       
    L = data & 0x01                    # T_LOW  -> H, TEMP < 10 deg
    H = (data & 0x02) >> 1             # T_HIGH -> H  TEMP > 64 deg
    C = (data & 0x04) >> 2             # T_CRIT -> H  TEMP > 80 deg
    return TEMP, C, H, L
```

###  Načítanie dát zo senzora 

```Python
# inicializacia rozhrania
from pyb import I2C
import time 

i2c = I2C(1, I2C.CONTROLLER, baudrate=100000)
```

Spustenie merania

```Python
def temp(n):
    for i in range(n):
        print(read_LM92_TR(i2c, 75))
        time.sleep(0.5)

temp(10)
```